# 3D Kooshball IR with a BrainWeb phantom

Same pipeline as `IR_TrueFISP_bssfp.ipynb` (cyclic 8-channel head coil +
isochromat / Bloch spin simulation + iterative NUFFT-SENSE reconstruction),
extended to the 3D kooshball inversion-recovery sequence on the BrainWeb
`subject04_crisp_v` phantom.

**Why the scan is simulated per TR segment:** every TR is an *independent*
inversion pulse + TI delay + 50 spokes block, and from the 2nd segment on the
system sits on the same TR cycle, so all segments produce the *identical*
contrast. The 7000-spoke scan simply repeats the same experiment 140 times.
Simulating `N_SEG_SIM` segments therefore gives the same k-space data (a
few tens of thousands of samples instead of ~900k) at a small fraction of
the spin-simulation cost.

**Runtime on an A100 (defaults below, 3 TIs):**
sequence generation ~1 min, simulation ~7 min/TI, reconstruction ~2-4 min/TI.
Set `N = 96` or `spin_count = 32` for a faster, coarser demo (the readout's
Nyquist must stay inside the grid, so keep `N >= N_x`).


In [ ]:
%matplotlib widget

import sys
import time

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system, load_brain_phantom

torch.set_default_device("cuda")
device = torch.device("cuda")


In [ ]:
# ----------------------------- sequence design ------------------------------
FOV = 250e-3            # m
N_x = 128               # samples per readout
N_channels = 8
alpha = 5.0             # excitation flip angle (deg)
slice_thickness = 3e-3  # m (sinc slice-select, kept from the 2D design)
TE = 3.98e-3
TR = 3500e-3
N_spokes = 7000
N_dummy = 20
spokes_per_TR = 50
rf_spoiling_inc = 117.0

# --------------------------- phantom / simulation ---------------------------
N = 128                 # simulation grid, N x N x N (keep N >= N_x)
spin_count = 64         # isochromats per voxel
N_SEG_SIM = 50          # TR segments to simulate per TI (2500 spokes; ~20 min each)

# T1 values of the brain phantom (Modules/phantom.py), in seconds:
T1_CSF, T1_GM, T1_WM = 4.0, 1.3, 0.9
TI_CSF = T1_CSF * np.log(2)   # ~2773 ms -> nulls CSF
TI_GM = T1_GM * np.log(2)     # ~901 ms  -> nulls GM
TI_WM = T1_WM * np.log(2)     # ~624 ms  -> nulls WM


In [ ]:
def main(
    FOV: float = 250e-3,
    N_x: int = 128,
    N_channels = 8,
    alpha: float = 5,
    slice_thickness: float = 3e-3,
    TE: float = 3.98e-3,
    TR: float = 3500e-3,
    TI: float = 2200e-3,
    N_spokes: int = 7000,
    N_dummy: int = 20,
    spokes_per_TR: int = 50,
    rf_spoiling_inc: float = 117,
    write_seq: bool = False,
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.pi,
        duration=10e-3,
        slice_thickness=slice_thickness,
        apodization=0.5,
        time_bw_product=4,
        system=system,
        return_gz=True,
        use="inversion",
    )

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
        use="excitation",
    )

    gx_base = pp.make_trapezoid(
        channel="x", flat_area=N_x * deltak, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=N_x,
        duration=gx_base.flat_time,
        delay=gx_base.rise_time,
        system=system,
    )
    gx_pre_base = pp.make_trapezoid(
        channel="x", area=-gx_base.area / 2, duration=2e-3, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=2e-3, system=system
    )
    gx_spoil_base = pp.make_trapezoid(
        channel="x", area=0.5 * N_x * deltak, system=system
    )
    gz_spoil = pp.make_trapezoid(channel="z", area=4 / slice_thickness, system=system)

    delay_TE = (
        np.ceil(
            (
                TE
                - pp.calc_duration(gx_pre_base)
                - gz.fall_time
                - gz.flat_time / 2
                - pp.calc_duration(gx_base) / 2
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )

    inv_duration = pp.calc_duration(rf_inv, gz_inv)
    delay_TI = (
        np.ceil(
            (TI - inv_duration / 2 - pp.calc_duration(gz) / 2) / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )

    N_segments = int(np.ceil(N_spokes / spokes_per_TR))

    # TR must fit: inversion + TI delay + spokes_per_TR GRE blocks
    gre_block_duration = (
        pp.calc_duration(gz)
        + pp.calc_duration(gx_pre_base)
        + delay_TE
        + pp.calc_duration(gx_base)
        + pp.calc_duration(gx_spoil_base, gz_spoil)
    )
    min_TR = inv_duration + delay_TI + spokes_per_TR * gre_block_duration
    delay_TR = np.ceil((TR - min_TR) / seq.grad_raster_time) * seq.grad_raster_time
    assert delay_TI > 0, "TI too short for inversion pulse"
    assert delay_TR >= 0, f"TR too short - minimum TR needed: {min_TR*1000:.0f}ms"

    rf_phase = 0
    rf_inc = 0
    spoke_counter = 0

    for seg in range(N_segments):
        # Inversion pulse at start of each segment
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI))

        for s in range(spokes_per_TR):
            i = seg * spokes_per_TR + s
            if i >= N_spokes + N_dummy:
                break

            rf.phase_offset = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_inc = (rf_inc + rf_spoiling_inc) % 360
            rf_phase = (rf_phase + rf_inc) % 360

            spoke_idx = i - N_dummy
            if spoke_idx >= 0:
                phi = np.arccos(1 - 2 * (spoke_idx + 0.5) / N_spokes)
                theta = np.pi * (1 + np.sqrt(5)) * spoke_idx
                kx = np.sin(phi) * np.cos(theta)
                ky = np.sin(phi) * np.sin(theta)
                kz = np.cos(phi)
            else:
                kx, ky, kz = 1, 0, 0

            seq.add_block(rf, gz)
            seq.add_block(
                pp.make_trapezoid(
                    channel="x",
                    area=gx_pre_base.area * kx,
                    duration=2e-3,
                    system=system,
                ),
                pp.make_trapezoid(
                    channel="y",
                    area=gx_pre_base.area * ky,
                    duration=2e-3,
                    system=system,
                ),
                pp.make_trapezoid(
                    channel="z",
                    area=gx_pre_base.area * kz + gz_reph.area,
                    duration=2e-3,
                    system=system,
                ),
            )
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(
                channel="x",
                amplitude=gx_base.amplitude * kx,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gy = pp.make_trapezoid(
                channel="y",
                amplitude=gx_base.amplitude * ky,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )
            gz_ro = pp.make_trapezoid(
                channel="z",
                amplitude=gx_base.amplitude * kz,
                flat_time=gx_base.flat_time,
                rise_time=gx_base.rise_time,
                fall_time=gx_base.fall_time,
                system=system,
            )

            if spoke_idx >= 0:
                seq.add_block(gx, gy, gz_ro, adc)
            else:
                seq.add_block(gx, gy, gz_ro)

            seq.add_block(
                pp.make_trapezoid(
                    channel="x", area=gx_spoil_base.area * kx, system=system
                ),
                pp.make_trapezoid(
                    channel="y", area=gx_spoil_base.area * ky, system=system
                ),
                pp.make_trapezoid(
                    channel="z",
                    area=gx_spoil_base.area * kz + gz_spoil.area,
                    system=system,
                ),
            )

        seq.add_block(pp.make_delay(delay_TR))

    ok, error_report = seq.check_timing()
    print(
        f"TI={TI*1000:.0f}ms - "
        + ("Timing check passed" if ok else f"Timing check failed: {error_report}")
    )

    seq.set_definition("FOV", [FOV, FOV, slice_thickness])
    seq.set_definition("Name", f"ir_kooshball_TI{int(TI*1000)}")

    if write_seq:
        seq.write(f"IR_Kooshball_TI{int(TI*1000)}ms.seq")

    return seq


t0 = time.time()
seq_csf = main(TI=TI_CSF, write_seq=True)
seq_gm = main(TI=TI_GM, write_seq=True)
seq_wm = main(TI=TI_WM, write_seq=True)
print(f"sequence generation: {time.time() - t0:.0f} s")


In [ ]:
# Brain phantom (BrainWeb subject04, resampled to N x N x N, FOV x FOV x FOV)
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(N, N, N),
    device=device,
    permute_order=(2, 1, 0),
)
print(f"phantom size (m): {[float(s) for s in phantom.size]}")

# Same coil model as the 2D IR TrueFISP notebook: cyclic 8-channel head coil
coil_sens = cyclic_head_coil(
    (N_channels, N, N, N),
    fov_scaling=(phantom.size[0] / FOV, phantom.size[1] / FOV, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sens.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)

data = phantom.build()
del phantom  # free the full 128^3 maps; only the sparse `data` is needed
print(f"simulation voxels (PD > 0): {data.PD.numel()}")


In [ ]:
from Modules.fast_iso import isochromat_sim_fast


def simulate_ir_segment(seq_file, data, N_SEG_SIM, spin_count):
    """Simulate N_SEG_SIM consecutive TR segments of the IR sequence.

    Each TR is an independent inversion + TI + spokes_per_TR block on the
    same cycle, so the segments carry the same contrast as the full scan.
    """
    cache_file = seq_file.replace(".seq", ".mr0cache.pt")
    try:
        mr0_seq = torch.load(cache_file, weights_only=False)
        print(f"loaded {seq_file} from cache: {len(mr0_seq)} reps")
    except Exception:
        t0 = time.time()
        mr0_seq = mr0.Sequence.import_file(seq_file, exact_trajectories=False)
        torch.save(mr0_seq, cache_file)
        print(f"imported {seq_file}: {time.time() - t0:.0f} s, {len(mr0_seq)} reps (cached)")

    # Inversion reps have no ADC and a ~180 deg pulse (dummy excitations
    # also have no ADC, but a small flip angle).
    inv_idxs = [i for i, r in enumerate(mr0_seq)
                if int(r.adc_usage.sum()) == 0 and abs(r.pulse.angle) > 3.0]
    reps_per_seg = spokes_per_TR + 1
    assert len(inv_idxs) >= N_SEG_SIM + 1
    start = inv_idxs[1]  # skip segment 0 (contains the dummy spokes)
    seg = mr0.Sequence(mr0_seq[start:start + reps_per_seg * N_SEG_SIM],
                       normalized_grads=False)
    n_adc = sum(int(r.adc_usage.sum()) for r in seg)
    print(f"simulating {N_SEG_SIM} segments: {n_adc} k-space samples")

    t0 = time.time()
    torch.manual_seed(0)
    signal = isochromat_sim_fast(seg, data.cuda(), spin_count=spin_count,
                                 spin_dist="rand", print_progress=True,
                                 chunk_bytes=2**30)
    print(f"simulation: {time.time() - t0:.0f} s")
    return signal.permute((1, 0)).contiguous(), seg  # (N_channels, N_samples)


signals, seg_seqs = {}, {}
for name, TI, seq_file in (
    ("csf", TI_CSF, f"IR_Kooshball_TI{int(TI_CSF*1000)}ms.seq"),
    ("gm", TI_GM, f"IR_Kooshball_TI{int(TI_GM*1000)}ms.seq"),
    ("wm", TI_WM, f"IR_Kooshball_TI{int(TI_WM*1000)}ms.seq"),
):
    print(f"=== {name} (TI = {TI*1000:.0f} ms) ===", flush=True)
    signals[name], seg_seqs[name] = simulate_ir_segment(seq_file, data, N_SEG_SIM,
                                                        spin_count)
    print(f"signal: {tuple(signals[name].shape)}")
    torch.save(signals[name].cpu(), f"IR_kooshball3D_{name}_signal.pt")


In [ ]:
class NonuniformFourierTransformOperator3D(LinearOperator):
    """F: image (N^3) -> multi-coil k-space (nC, nK).

    3D analogue of the operator used in the 2D IR TrueFISP notebook.
    ``k`` is given in cycles per voxel (k in 1/m times the voxel size dv = FOV/N;
    the NUFFT applies the 2*pi itself).
    Note: the Plan.py gridding evaluates the DFT at -k, so the sign is
    flipped here to match the spin-simulation signal S(k) = sum M(x) e^{-2*pi*i*k.x}
    (the 2D radial trajectory is symmetric under k -> -k, which is why the
    2D notebook never noticed).
    """

    def __init__(self, k, input_shape, device=None):
        nD, nK = k.shape                      # (3, nK)
        nC, nX, nY, nZ = input_shape
        self.n_modes = (nX, nY, nZ)
        self.k = (-k).unsqueeze(-1)           # (3, nK, 1)
        self.forward_shape = (nC, nX, nY, nZ, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


def reconstruct_ir(signal, seq, maxiter=100):
    """Iterative NUFFT + SENSE reconstruction (CG on A^H A), 3D."""
    ktraj = torch.tensor(seq.get_kspace(), dtype=torch.float32)[:, :3].T.contiguous()
    k = (ktraj * (FOV / N)).to(device)       # cycles per voxel (NUFFT convention)

    F = NonuniformFourierTransformOperator3D(k, (N_channels, N, N, N))
    C = ChannelOperator(coil_sens, (N, N, N))
    A = F @ C

    b = signal.to(device).contiguous().view(torch.float32).ravel()
    Q = A.H @ A
    AHd = A.H @ b
    x = torch.zeros(Q.shape[1], dtype=torch.float32)
    x_recon = conjugate_gradient(Q, AHd, x, maxiter=maxiter)
    return x_recon.view(torch.complex64).reshape(N, N, N)


images = {}
for name, TI in (("csf", TI_CSF), ("gm", TI_GM), ("wm", TI_WM)):
    t0 = time.time()
    images[name] = reconstruct_ir(signals[name], seg_seqs[name])
    print(f"{name}: reconstructed in {time.time() - t0:.0f} s, "
          f"{tuple(images[name].shape)}")


In [ ]:
mid = N // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, TI, label) in zip(
    axes,
    (("csf", TI_CSF, "CSF"), ("gm", TI_GM, "GM"), ("wm", TI_WM, "WM")),
):
    img = np.rot90(images[name].abs().cpu().numpy(), k=3, axes=(0, 1))
    ax.imshow(img[:, :, mid], cmap="gray")
    ax.set_title(f"{label} nulled  (TI={TI*1000:.0f} ms)")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Maximum intensity projection as a second view
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, TI, label) in zip(
    axes,
    (("csf", TI_CSF, "CSF"), ("gm", TI_GM, "GM"), ("wm", TI_WM, "WM")),
):
    img = np.rot90(images[name].abs().cpu().numpy(), k=3, axes=(0, 1))
    ax.imshow(img.max(axis=2), cmap="gray")
    ax.set_title(f"{label} nulled  (MIP)")
    ax.axis("off")
plt.tight_layout()
plt.show()
